# Monte Carlo Analysis of Conductance Variability

This notebook performs Monte Carlo simulations to analyze how conductance variability affects neuronal firing patterns and to identify correlations in the conductance space.

---

# *This file allows to generate all plots necessary for figures*

# **Useful packages and functions**

## Dependencies and Setup

In [ ]:
using DifferentialEquations, Plots, Polynomials, LaTeXStrings, ColorSchemes, DelimitedFiles, DataFrames
using Statistics, StatsPlots, Random, ProgressMeter, Printf, LinearAlgebra, Plots.PlotMeasures
include("DA_kinetics.jl") # Loading of DA kinetics of gating variables
include("DA_models.jl") # Loading of DA model
include("DA_utils.jl"); # Loading of some utils functions

# **Global variables**

## Model Parameters

In [ ]:
# Definition of simulation time (in ms)
const Tfinal = 20000
const tspan  = (0.0, Tfinal)
tt = 0. : 0.01 : Tfinal
tt_rand = 0. : 1 : Tfinal

# Definition of reversal potential values (in mV), [Mg] and membrane capacitance
const VNa     = 60. # Sodium reversal potential
const VK      = -90. # Potassium reversal potential
const VCa     = 50. # Calcium reversal potential
const VH      = -29. # H reversal potential
const VLNS    = -65. # Leak reversal potential
const EPacemaker = 4.2732015978991615 # Reversal potential of pacemaking channels

const C       = 1. # Membrane capacitance
const fCa     = 0.018 # Fraction of unbuffered free calcium
const ICapmax = 11 # Maximum calcium pump current
const F       = 96520 # Faraday constant in ms*µA/mmol (and taking cm³=mL)
const d       = 15 # Soma diameter in cm
const L       = 25 # Soma length

# Definition of voltage range for the DICs
const Vmin = -100 
const Vmax = 50
const Vrange = range(Vmin, stop=Vmax, step=0.0154640);

## Plotting Configuration

In [ ]:
# Modifying backend GR attributes
gr(guidefontsize=25, tickfontsize=15, legendfontsize=12, margin=5Plots.mm, grid=false)
myApple = RGBA(187/255, 206/255, 131/255, 1)
mySalmon = RGBA(243/255, 124/255, 130/255)
myYellow = RGBA(228/255, 205/255, 121/255, 1)
myBlue = RGBA(131/255, 174/255, 218/255, 1)
myDarkBlue = RGBA(114/255, 119/255, 217/255, 1)
myOrange = RGBA(241/255, 175/255, 113/255, 1)
myPink = RGBA(243/255, 124/255, 130/255, 1)
myPurple = RGBA(169/255, 90/255, 179/255, 1)
myGreen = RGBA(132/255, 195/255, 168/255, 1)
myRed = RGBA(158/255, 3/255, 8/255, 1)
myGray = RGBA(150/255, 150/255, 150/255, 1)
myLightBlue = RGBA(127/255, 154/255, 209/255, 1);
default(fmt = :png);

In [ ]:
# Define a struct (optional, but useful if you need parameters)
struct NoisyFunction
    amplitude::Float64  # amplitude of the noise
end

# Overload the () operator to make the struct callable
function (nf::NoisyFunction)(x::Float64)
    noise = nf.amplitude * randn()  # Generate Gaussian noise (mean 0, std 1)
    return noise  # Example function with noise
end

function condition(u,t,integrator) # Event when event_f(u,t) == 0
  (u[1]- (-20.))
end

function affect!(integrator)
end

cb = ContinuousCallback(condition, affect!, nothing, save_positions = (true, false));

# Simu true model with pacemaker

In [ ]:
# Initializing some variables
N = 200
# g_all_pacemaker = zeros(N, 10)
g_all_pacemaker = readdlm("./data/g_all_pacemaker.dat")

# Input current definition
Iapp(t) = 0 # pA

# Initial conditions
V0 = -50.
Ca0 = 1e-4
x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
    0., 0., mH_inf(V0), Ca0]

n_neurons = 0
while n_neurons < N
    gNa        = 25. *2*rand()[1] # Sodium current maximal conductance
    gCaL       = 1. *2*rand()[1] # L-type calcium current maximal conductance
    gKd        = 10. *2*rand()[1] # Delayed-rectifier potassium current maximal conductance
    gKA        = 1.68 *2*rand()[1] # A-type potassium current maximal conductance
    gKERG      = 0.13 *2*rand()[1] # ERG current maximal conductance
    gKSK       = 0.3 *2*rand()[1] # SK current maximal conductance
    gH         = 0.078 *2*rand()[1] # H current maximal conductance
    gLNS       = 0.01 *2*rand()[1] # Leak non specific current maximal conductance
    gLCa       = 0.00245 *2*rand()[1] # Leak calcium current maximal conductance
    gPacemaker = 20 *2*rand()[1] # Pacemaker current maximal conductance
    p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gPacemaker)

    # Simulation
    prob = ODEProblem(DA_ODE_true_instant, x0, tspan, p) # Describing the problem
    sol = solve(prob; maxiters=1e6); # Solving the problem
    sol_spike_times = solve(prob; callback=cb, save_everystep=false,save_start=false,save_end=false)

    x         = sol(tt)
    V_plot    = x[1, 500000:end]

    if length(sol_spike_times.t) < 20
        continue
    else
        # Extracting ISIs
        spike_times = sol_spike_times.t
        filter!(x -> x ≥ 5000, spike_times)

        if length(spike_times) < 5
            continue
        end
        
        # Extracting ISIs
        ISIs = zeros(length(spike_times)-1)
        for k = 1 : length(spike_times) - 1
            ISIs[k] = spike_times[k+1] - spike_times[k]
        end

        f = 1000/mean(ISIs)

        if f < 5 && f > 1 && maximum(V_plot) < 30 && maximum(V_plot) > 10 && minimum(V_plot) < -70 && minimum(V_plot) > -90
            n_neurons = n_neurons + 1
            display(n_neurons)
            display(f)
            display([gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gPacemaker])
            g_all_pacemaker[n_neurons, :] = [gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa, gPacemaker]

            voltage = plot(tt[500000:end]./1e3, V_plot, linewidth=2.5, color=myDarkBlue, xlims=(19., 20.),
                legend=false, ylims=(-100, 30), margins=20Plots.px, size=(1200, 800))
            ylabel!("V (mV)")

            display(voltage)
        end
    end
end

In [ ]:
writedlm("./data/g_all_pacemaker.dat", g_all_pacemaker);

# Simu original model

In [ ]:
# Initializing some variables
N = 200
g_all = zeros(N, 9)

# Input current definition
Iapp(t) = 0 # pA

# Initial conditions
V0 = -50.
Ca0 = 1e-4
x0 = [V0, m_inf(V0), h_inf(V0), hs_inf(V0), l_inf(V0), n_inf(V0), p_inf(V0), q1_inf(V0), q2_inf(V0), 
    0., 0., mH_inf(V0), Ca0]

n_neurons = 0
while n_neurons < N
    gNa        = 25. *2*rand()[1] # Sodium current maximal conductance
    gCaL       = 1. *2*rand()[1] # L-type calcium current maximal conductance
    gKd        = 10. *2*rand()[1] # Delayed-rectifier potassium current maximal conductance
    gKA        = 1.68 *2*rand()[1] # A-type potassium current maximal conductance
    gKERG      = 0.13 *2*rand()[1] # ERG current maximal conductance
    gKSK       = 0.3 *2*rand()[1] # SK current maximal conductance
    gH         = 0.078 *2*rand()[1] # H current maximal conductance
    gLNS       = 0.01 *2*rand()[1] # Leak non specific current maximal conductance
    gLCa       = 0.00245 *2*rand()[1] # Leak calcium current maximal conductance
    p = (Iapp, gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa)

    # Simulation
    prob = ODEProblem(DA_ODE, x0, tspan, p) # Describing the problem
    sol = solve(prob; maxiters=1e6, verbose=false); # Solving the problem
    sol_spike_times = solve(prob; callback=cb, save_everystep=false,save_start=false,save_end=false, verbose=false)

    x         = sol(tt)
    V_plot    = x[1, 500000:end]

    if length(sol_spike_times.t) < 20
        continue
    else
        # Extracting ISIs
        spike_times = sol_spike_times.t
        filter!(x -> x ≥ 5000, spike_times)

        if length(spike_times) < 5
            continue
        end
        
        # Extracting ISIs
        ISIs = zeros(length(spike_times)-1)
        for k = 1 : length(spike_times) - 1
            ISIs[k] = spike_times[k+1] - spike_times[k]
        end

        f = 1000/mean(ISIs)

        if f < 5 && f > 1 && maximum(V_plot) < 30 && maximum(V_plot) > 0 && minimum(V_plot) < -60 && minimum(V_plot) > -90
            n_neurons = n_neurons + 1
            display(n_neurons)
            display(f)
            display([gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa])
            g_all[n_neurons, :] = [gNa, gCaL, gKd, gKA, gKERG, gKSK, gH, gLNS, gLCa]

            voltage = plot(tt[500000:end]./1e3, V_plot, linewidth=2.5, color=myDarkBlue, xlims=(19., 20.),
                legend=false, ylims=(-100, 30), margins=20Plots.px, size=(1200, 800))
            ylabel!("V (mV)")

            display(voltage)
        end
    end
end

In [ ]:
writedlm("./data/g_all.dat", g_all);

In [ ]:
N = 200
g_all = zeros(N, 9)
for i = 1 : N
    filename = "./data/g_all$(i).dat"
    g_all[i, :] = vec(readdlm(filename))
end

writedlm("./data/g_all.dat", g_all);

# **Scatter plots**

In [ ]:
g_all = readdlm("./data/g_all.dat")
g_all_pacemaker = readdlm("./data/g_all_pacemaker.dat")
maxs = [100, 5, 50, 5, 1, 2, 0.5, 0.1, 0.1, 75]
names = ["gNa", "gCaL", "gKd", "gKA", "gKERG", "gKSK", "gH", "gLNS", "gLCa", "gPace"];

In [ ]:
using Plots, StatsPlots, DelimitedFiles

# Load your data
g_all = readdlm("./data/g_all.dat")
g_all_pacemaker = readdlm("./data/g_all_pacemaker.dat")

# Number of columns (assuming g_all has 9 columns, g_all_pacemaker might have 10 with gPace)
n_cols = size(g_all, 2)

# Create subplots for each column
plots = []

for i in 1:n_cols
    # Extract column data
    data_all = g_all[:, i]
    
    # Check if g_all_pacemaker has this column
    if i <= size(g_all_pacemaker, 2)
        data_pacemaker = g_all_pacemaker[:, i]
    else
        # If pacemaker data doesn't have this column, create empty data
        data_pacemaker = Float64[]
    end
    
    # Create violin plot for current column
    p = violin([1], [data_all], 
               side=:left, 
               label="g_all", 
               color=:lightblue,
               alpha=0.7,
               ylabel="g",
               xlims=(0.5, 1.5),
               xticks=([1], [names[i]]),
               ylims=(0, maxs[i]),
               yticks=([0, maxs[i]]))
    
    # Add right side violin if pacemaker data exists
    if !isempty(data_pacemaker)
        violin!(p, [1], [data_pacemaker], 
                side=:right, 
                label="g_all_pacemaker", 
                color=:lightcoral,
                alpha=0.7)
    end
    
    push!(plots, p)
end

i=10
# Create violin plot for current column
p = violin([1], [g_all_pacemaker[:, 10]], 
           side=:right, 
           label="g_all", 
           color=:lightcoral,
           alpha=0.7,
           ylabel="g",
           xlims=(0.5, 1.5),
           xticks=([1], [names[i]]),
           ylims=(0, maxs[10]),
           yticks=([0, maxs[10]]))


push!(plots, p)

# Arrange all plots in a grid
final_plot = plot(plots..., 
                  layout=(2, 5),  # 2 rows, 5 columns (adjust as needed)
                  size=(1200, 800),
                  titlefontsize=10,
                  legendfontsize=8,
                  margin=5Plots.mm, legend=false)

# Display the plot
display(final_plot)

# Optional: Save the plot
# savefig(final_plot, "./figures/violin_plots_degenerate.pdf")
# savefig(final_plot, "./figures/violin_plots_comparison.svg")

In [ ]:
using LinearAlgebra, Plots, DelimitedFiles, Statistics
using MultivariateStats: PCA, fit, transform, principalvars, projection

# Load your data
g_all_pacemaker = readdlm("./data/g_all_pacemaker.dat")

# Your parameter names (adjust based on actual columns in g_all_pacemaker)
names = ["gNa", "gCaL", "gKd", "gKA", "gKERG", "gKSK", "gH", "gLNS", "gLCa", "gPace"]

println("Original data shape: ", size(g_all_pacemaker))
println("Number of samples: ", size(g_all_pacemaker, 1))
println("Number of features: ", size(g_all_pacemaker, 2))

# Transpose data for MultivariateStats (features × samples)
X = g_all_pacemaker'  # Now features are rows, samples are columns

# Optional: Standardize the data (recommended for PCA)
# This centers and scales each feature to have mean=0 and std=1
X_standardized = (X .- mean(X, dims=2)) ./ std(X, dims=2)

# Perform PCA
pca_model = fit(PCA, X_standardized)

# Transform data to principal component space
Y = transform(pca_model, X_standardized)

# Get explained variance ratio
explained_var = principalvars(pca_model)
total_var = sum(explained_var)
explained_var_ratio = explained_var ./ total_var
cumulative_var = cumsum(explained_var_ratio)

println("\n=== PCA Results ===")
println("Explained variance ratio for each PC:")
for i in 1:min(length(explained_var_ratio), 10)
    println("PC$i: $(round(explained_var_ratio[i]*100, digits=2))%")
end

println("\nCumulative explained variance:")
for i in 1:min(length(cumulative_var), 10)
    println("PC1-PC$i: $(round(cumulative_var[i]*100, digits=2))%")
end

# Plot 1: Scree plot (explained variance)
p1 = bar(1:min(length(explained_var_ratio), 10), 
         explained_var_ratio[1:min(length(explained_var_ratio), 10)] .* 100,
         title="Scree Plot - Explained Variance",
         xlabel="Principal Component",
         ylabel="Explained Variance (%)",
         legend=false,
         color=:steelblue)

# Plot 2: Cumulative explained variance
p2 = plot(1:min(length(cumulative_var), 10), 
          cumulative_var[1:min(length(cumulative_var), 10)] .* 100,
          title="Cumulative Explained Variance",
          xlabel="Principal Component",
          ylabel="Cumulative Explained Variance (%)",
          marker=:circle,
          linewidth=2,
          legend=false,
          color=:darkred)
hline!([80, 95], linestyle=:dash, color=[:orange, :red], alpha=0.7)

# Plot 3: PC1 vs PC2 scatter plot
p3 = scatter(Y[1, :], Y[2, :],
             title="PCA: PC1 vs PC2",
             xlabel="PC1 ($(round(explained_var_ratio[1]*100, digits=1))%)",
             ylabel="PC2 ($(round(explained_var_ratio[2]*100, digits=1))%)",
             alpha=0.6,
             markersize=3,
             color=:purple,
             legend=false)

# Plot 4: Loadings plot for PC1 and PC2
loadings = projection(pca_model)  # This gives the loadings matrix
n_features = size(loadings, 1)
feature_names = names[1:min(n_features, length(names))]

p4 = scatter(loadings[:, 1], loadings[:, 2],
             title="PCA Loadings: PC1 vs PC2",
             xlabel="PC1 Loading",
             ylabel="PC2 Loading",
             alpha=0.7,
             markersize=6,
             color=:darkgreen,
             legend=false)

# Add feature labels to loadings plot
for i in 1:n_features
    annotate!(p4, loadings[i, 1], loadings[i, 2], 
              text(feature_names[i], 8, :center))
end

# Combine all plots
combined_plot = plot(p1, p2, p3, p4, 
                     layout=(2, 2), 
                     size=(1000, 800),
                     titlefontsize=10)

display(combined_plot)
savefig(combined_plot, "./figures/PCA.pdf")
savefig(combined_plot, "./figures/PCA.svg")

# Print loadings matrix for interpretation
println("\n=== PC Loadings (first 3 components) ===")
println("Feature\t\tPC1\t\tPC2\t\tPC3")
println("-" ^ 50)
for i in 1:n_features
    pc1_load = round(loadings[i, 1], digits=3)
    pc2_load = round(loadings[i, 2], digits=3)
    pc3_load = i <= size(loadings, 2) && size(loadings, 2) >= 3 ? round(loadings[i, 3], digits=3) : 0.0
    println("$(feature_names[i])\t\t$pc1_load\t\t$pc2_load\t\t$pc3_load")
end

# Optional: Save transformed data
# Transform back to samples × features format for easier interpretation
Y_transposed = Y'  # Now samples are rows, PCs are columns
println("\nTransformed data shape: ", size(Y_transposed))

# You can save the PC scores if needed
# writedlm("pca_scores.dat", Y_transposed)

# Return useful objects for further analysis
pca_results = (
    model = pca_model,
    transformed_data = Y_transposed,
    loadings = loadings,
    explained_variance_ratio = explained_var_ratio,
    cumulative_variance = cumulative_var
)

println("\nPCA analysis complete! Results stored in 'pca_results' named tuple.")

In [ ]:
s = scatter(size=(1200, 800), margins=20Plots.px)
i1 = 1
i2 = 2
scatter!(g_all[:, i1], g_all[:, i2], legend=false, color=myDarkBlue, markersize=10)
scatter!(g_all_pacemaker[:, i1], g_all_pacemaker[:, i2], legend=false, color=myGray, markersize=10)
xlims!((0, maxs[i1]))
ylims!((0, maxs[i2]))
xlabel!(names[i1])
ylabel!(names[i2])
display(s)
# savefig(s, "./figures/degeneracy.pdf")
# savefig(s, "./figures/degeneracy.svg")

In [ ]:
s = scatter(size=(1200, 800), margins=20Plots.px)
i1 = 1
i2 = 3
scatter!(g_all[:, i1], g_all[:, i2], legend=false, color=myDarkBlue, markersize=10)
scatter!(g_all_pacemaker[:, i1], g_all_pacemaker[:, i2], legend=false, color=myGray, markersize=10)
xlims!((0, maxs[i1]))
ylims!((0, maxs[i2]))
xlabel!(names[i1])
ylabel!(names[i2])
display(s)
# savefig(s, "./figures/degeneracy2.pdf")
# savefig(s, "./figures/degeneracy2.svg")

In [ ]:
s = scatter(size=(1200, 800), margins=20Plots.px)
i1 = 2
i2 = 3
scatter!(g_all[:, i1], g_all[:, i2], legend=false, color=myDarkBlue, markersize=10)
scatter!(g_all_pacemaker[:, i1], g_all_pacemaker[:, i2], legend=false, color=myGray, markersize=10)
xlims!((0, maxs[i1]))
ylims!((0, maxs[i2]))
xlabel!(names[i1])
ylabel!(names[i2])
display(s)
# savefig(s, "./figures/degeneracy3.pdf")
# savefig(s, "./figures/degeneracy3.svg")

In [ ]:
for i1 = 1 : 9
    for i2 = 1 : i1 - 1
        sij = scatter(g_all[:, i1], g_all[:, i2], legend=false, color=myDarkBlue, markersize=5)
        scatter!(g_all_pacemaker[:, i1], g_all_pacemaker[:, i2], legend=false, color=myGray, markersize=5)
        xlims!((0, maxs[i1]))
        ylims!((0, maxs[i2]))
        xlabel!(names[i1])
        ylabel!(names[i2])
        display(sij)
    end
end

In [ ]:
for i1 = 1 : 9
    i2 = 10
    sij = scatter(g_all_pacemaker[:, i1], g_all_pacemaker[:, i2], legend=false, color=myGray, markersize=5)
    xlims!((0, maxs[i1]))
    ylims!((0, maxs[i2]))
    xlabel!(names[i1])
    ylabel!(names[i2])
    display(sij)
end

In [ ]:
scatter(g_all_pacemaker[:, 1], g_all_pacemaker[:, 3], zcolor=g_all_pacemaker[:, 10], markersize=5, label=false, grid=true, camera=(30, 10))
scatter!(g_all[:, 1], g_all[:, 3], color=myDarkBlue, markersize=5, label=false, grid=true, camera=(30, 10))

In [ ]:
scatter(g_all_pacemaker[:, 1], g_all_pacemaker[:, 3], g_all_pacemaker[:, 10], markersize=5, color=myGray, label=false, size=(1000, 1000), grid=true, camera=(30, 10))
xlabel!(names[1])
ylabel!(names[3])
zlabel!(names[10])
scatter!(g_all[:, 1], g_all[:, 3], g_all_pacemaker[:, 10], markersize=5, color=myDarkBlue, label=false, size=(1000, 1000), grid=true, camera=(40, 10))
# plot!(line_points[:, 1], line_points[:, 2], line_points[:, 3], lw=2, label="Regression Line")